# Experiment with Overlapping Chunks

In [149]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from ipywidgets import interact

In [ ]:
def import_tokens(src_id):
    src_path = f"../{src_id}"
    token_file = f"{src_path}/{src_id}-TOKEN.csv"
    TOKEN = pd.read_csv(token_file)
    idx_offset = TOKEN.columns.to_list().index('token_str')
    ohco = TOKEN.columns.to_list()[:idx_offset]
    return TOKEN.set_index(ohco)

def chunk_tokens(TOKEN, chunk_size=150, overlap=20, min_len=50):
    """Split text into overlapping word-level chunks."""
    words = TOKEN.term_str.dropna().to_list()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.split()) >= min_len:  # drop tiny tail chunks
            chunks.append(chunk)
    return chunks

def get_nmf_topics(nmf_model, tfidf_vectorizer, n_top_words):
    words = tfidf_vectorizer.get_feature_names_out()
    topic_words = {}
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_word_indices = topic.argsort()[: -n_top_words - 1 : -1]
        topic_words[f"Topic {topic_idx}"] = [words[i] for i in top_word_indices]    
    return pd.DataFrame(topic_words)

In [284]:
SOURCES = {
    'ajtzibab'           : dict(lang='quc'),
    'christenson'        : dict(lang='quc'),
    'colop'              : dict(lang='quc'),
    'christenson_ximenez': dict(lang='quc'),
    'ximenez'          : dict(lang='quc'),
    'recinos'          : dict(lang='spa'),
    'tedlock'          : dict(lang='eng'),
}


In [ ]:
@interact(
    min_df = (1, 20, 1),
    max_df = (.1, 1, .01),
    src_id = SOURCES.keys(), 
    n_topics = (2, 20, 1),
    chunk_size = (100, 1500, 10), 
    overlap = (0., .9, .1)
)

def plot_text(
        src_id = 'colop', 
        chunk_size = 1000, 
        overlap = .9,
        min_df = 5,
        max_df = .35,
        n_topics = 8
    ):

    # Create F1 data
    overlap_int = int(overlap * chunk_size)
    TOKEN = import_tokens(src_id)
    F1 = chunk_tokens(TOKEN, chunk_size=chunk_size, overlap=overlap_int)

    # Create Count matrix
    count_engine = TfidfVectorizer(lowercase=True, 
        max_df=max_df, 
        min_df=min_df, 
        strip_accents=None,
        norm='l2')
    X = count_engine.fit_transform(F1)

    # Create topic model
    topic_engine = NMF(
        n_components=n_topics,
        init='nndsvd'
    )
    THETA = pd.DataFrame(topic_engine.fit_transform(X))
    THETA.index.name = 'chunk_id'
    THETA.columns.name = 'topic_id'

    # Get topics
    TOPICS = get_nmf_topics(topic_engine, count_engine, 7)
    print(TOPICS.T)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(20,4))
    sns.heatmap(THETA.T, cmap="YlGnBu")
    plt.title(f"{src_id.replace('_', ' ').title()}", fontdict={'size':20}, y=1.01)
    plt.xlabel("Synagm", fontdict={'size': 16})
    plt.ylabel("Paradigm", fontdict={'size': 16})
    plt.show()

interactive(children=(Dropdown(description='src_id', index=2, options=('ajtzibab', 'christenson', 'colop', 'ch…